In [1]:
# networks.py

# %% Dependencies
import torch 
import torch.distributions as td
import torch.nn as nn 
from torch.multiprocessing import Pool
#from torch_geometric.utils import dense_to_sparse

import pygmtools as pygm
# TODO: check padding method. 
from pygmtools.utils import dense_to_sparse
pygm.set_backend('pytorch')

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import get_edge_type

# %%
class ContNodeFeats(nn.Module):
    def __init__(self, node_size, cont_node_feat, batch_size):
        super(ContNodeFeats, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.batch_size = batch_size

        self.cont_node_feat_1 = nn.Linear(self.node_size, 
            self.cont_node_feat * self.node_size)

    def forward(self):
        Z = torch.randn((self.batch_size, self.node_size))
        X = self.cont_node_feat_1(Z)
        X = X.view(self.batch_size, self.node_size, self.cont_node_feat)
        return X

# %%
class DisNodeFeat(nn.Module): 
    def __init__(self, node_size, cell_types, batch_size):
        super(DisNodeFeat, self).__init__()
        self.node_size = node_size
        self.cell_types = cell_types
        self.batch_size = batch_size

        self.cell_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((1, self.cell_types))
            )
        )

    def forward(self):
        self.dist = td.Categorical(logits=self.cell_logits)
        self.sample = self.dist.sample(
            (self.batch_size, self.node_size)).squeeze(2)
        self.logLik = self.dist.log_prob(self.sample).sum(dim=1)
        return self.sample + 1, self.logLik # category indexing starts at 1. 

# %%
class AdjacencyMatrix(nn.Module):
    def __init__(self, node_size, batch_size):
        super(AdjacencyMatrix, self).__init__()
        self.node_size = node_size
        self.batch_size = batch_size

        self.edge_logits = nn.Parameter(
            nn.init.xavier_normal_( # Glorot initialization. 
                torch.empty((self.node_size, self.node_size))
            )
        )

    def forward(self):
        self.dist = td.Bernoulli(logits=self.edge_logits)
        self.sample = self.dist.sample([self.batch_size])#.squeeze()
        self.logLik = self.dist.log_prob(self.sample).sum(dim=(1,2))
        return self.sample, self.logLik

# %% 
class EdgeFeats(nn.Module):
    def __init__(self, node_size, cont_edge_feat, batch_size):
        super(EdgeFeats, self).__init__() 
        self.node_size = node_size
        self.cont_edge_feat = cont_edge_feat
        self.batch_size = batch_size

    @staticmethod
    def get_edge_type_app(A, C_x):
        # A: [b, 2, #edges] C: [b, #nodes].
        edges = list(zip(A.long()[0], A.long()[1]))
        e_c = list(map(lambda x: get_edge_type(x, C_x.int()), edges))
        e_c = torch.tensor(e_c).unsqueeze(-1)
        return e_c
        
    def forward(self, A, C_x): 
        # Discrete Edge Features:
        with torch.no_grad():
            with Pool() as pool: 
                e_c = torch.stack(
                    pool.starmap(EdgeFeats.get_edge_type_app, 
                               zip(A.unbind(), C_x.unbind()))
                ) 
                               
        # Continuous Edge Features
        Z = torch.randn((self.batch_size, self.node_size**2, 1))
        W = torch.normal(mean=0, std=1, 
                         size=(self.batch_size, 1, self.cont_edge_feat))
        E = torch.bmm(Z, W)
        E = E[:, :A.shape[2], :] # match number of edges.

        # Combines continuous and discrete node features.
        edge_features = torch.cat((e_c, E), dim=-1) 
        return edge_features


# %% Full Model 
class EGG(nn.Module):
    def __init__(self, node_size, cont_node_feat, cell_types, cont_edge_feat,
                 batch_size=1): 
        super(EGG, self).__init__()
        self.node_size = node_size
        self.cont_node_feat = cont_node_feat       
        self.cell_types = cell_types
        self.cont_edge_feat = cont_edge_feat
        self.batch_size = batch_size

        # Sub-Generator models.
        self.ContNodeFeats = ContNodeFeats(self.node_size, self.cont_node_feat,
                                           self.batch_size)
        self.DisNodeFeat = DisNodeFeat(self.node_size, self.cell_types,
                                       self.batch_size)
        self.AdjacencyMatrix = AdjacencyMatrix(self.node_size, self.batch_size)
        self.EdgeFeats = EdgeFeats(self.node_size, self.cont_edge_feat, 
                                   self.batch_size)

    def forward(self):
        # Sub-Generator models.
        X = self.ContNodeFeats()

        C_x, C_x_logLik = self.DisNodeFeat()

        A, A_logLik = self.AdjacencyMatrix()
        A = dense_to_sparse(A)[0].transpose(1, 2)

        E = self.EdgeFeats(A, C_x)
        
        return X, C_x, A, E, C_x_logLik, A_logLik 

In [18]:
# trainer.py

# %% Dependencies:
import torch 
import torch.nn as nn

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData

import time

class Trainer:
    def __init__(self, 
                 generator: EGG, 
                 explainee: nn.Module, 
                 optimizer: torch.optim.Optimizer, 
                 criterion: callable) -> None:
        self.generator = generator
        self.explainee = explainee
        self.criterion = criterion
        self.optimizer = optimizer

    def train(self, target, num_epochs):
        for epoch in range(num_epochs):
            start = time.time()

            self.optimizer.zero_grad()
            # X[b, nodes, feat]; C_x[b, nodes]; A[b, 2, edges];
            # E[b, edges, feat]; lik[b]
            X, C_x, A, E, C_x_logLik, A_logLik = self.generator()
            
            graph_list = [
                NucleiData(X, C_x, A, E) for (X, C_x, A, E) in 
                zip(X.unbind(), C_x.unbind(), A.unbind(), E.unbind())
            ]

            graph_list = [
                clear_iso_nodes(graph) for graph in graph_list
            ]

            pred_loss = PredLoss(target, self.criterion, self.explainee)

            pred_losses = pred_loss(graph_list) 
            
            loss = (pred_losses.sum() + 
                    pred_losses @ C_x_logLik + 
                    pred_losses @ A_logLik) / self.generator.batch_size

            loss.backward()
            self.optimizer.step()

            end = time.time()

            print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.10f}" + 
                f"Time: {end-start:.2f} seconds")

In [19]:
# losses.py

# %% Dependencies:
import torch 
import torch.nn as nn 

# %%
class PredLoss(nn.Module):
    def __init__(self, target, criterion, explainee):
        super(PredLoss, self).__init__()
        self.target = target
        self.criterion = criterion
        self.explainee = explainee

    def pred_loss_fn(self, example):
        try: 
            example.to(torch.device(0))
            explainee_pred = torch.softmax(self.explainee(example), dim=0).cpu()

            return self.criterion(explainee_pred, self.target)
        
        except Exception as e:
            return self.criterion(torch.tensor([0.5, 0.5]), self.target)


    def forward(self, examples):
        pred_losses = torch.stack([
            self.pred_loss_fn(example) for example in examples
        ])

        return pred_losses

In [20]:
# utils.py

# %% Dependencies
from typing import Optional
import copy 
from torch_geometric.utils import remove_isolated_nodes

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData

# %%
def clear_iso_nodes(example: NucleiData, 
                    num_nodes: Optional[int] = None) -> NucleiData: 
    edge_index, edge_attr, mask = (
        remove_isolated_nodes(example.edge_index, 
                              example.edge_attr, 
                              num_nodes=num_nodes)
    )
    example_masked = NucleiData(
        x = example.x[mask], 
        edge_index = edge_index, 
        cell_type = example.cell_type[mask], 
        edge_attr = edge_attr,
    )

    return example_masked

## tests

In [21]:
%%time

import random 
import time
import pickle

import sys 
sys.path.append("../scripts/ceograph/")
from ceograph import NucleiData, load_model

# Load ad obs 
# path = "../data/slides/LUDA/ad_train_nx_100.pkl"

# with open(path, 'rb') as f:
#     ad_train_nx_100 = pickle.load(f)

# %% Training Config

# Reproducibility 
random.seed(0)
torch.manual_seed(0)
device = torch.device(0)

# Model to be explained. 
explainee = load_model(path = "../data/trained/epoch_263.pt",
                       device=device)

# Model Parameters:
max_nodes = 50
cont_node_feats = 11
cell_types = 5
cont_edge_feat = 2

EGG_Model = EGG(node_size=max_nodes, cont_node_feat=cont_node_feats, 
                cell_types=cell_types, cont_edge_feat=cont_edge_feat, 
                batch_size=100)

CPU times: user 6.88 ms, sys: 1.98 ms, total: 8.86 ms
Wall time: 7.98 ms


In [23]:
%%time

from torch.optim import Adam

optimizer = Adam(EGG_Model.parameters())

trainer = Trainer(EGG_Model, explainee, optimizer, nn.BCELoss())

trainer.train(torch.tensor([0.0, 1.0]), 5)

called
Epoch [1/5], Loss: -8658.1279296875Time: 3.78 seconds
called
Epoch [2/5], Loss: -9269.9765625000Time: 3.86 seconds
called
Epoch [3/5], Loss: -7308.4121093750Time: 3.97 seconds
called
Epoch [4/5], Loss: -8513.6269531250Time: 3.93 seconds
called
Epoch [5/5], Loss: -9041.3183593750Time: 3.87 seconds
CPU times: user 36.4 s, sys: 16.2 s, total: 52.6 s
Wall time: 19.4 s


In [13]:
%%time


X, C_x, A, E, C_x_logLik, A_logLik = EGG_Model()

print(X.shape, C_x.shape, A.shape, E.shape, C_x_logLik.shape, A_logLik.shape)



torch.Size([1000, 50, 11]) torch.Size([1000, 50]) torch.Size([1000, 2, 1337]) torch.Size([1000, 1337, 3]) torch.Size([1000]) torch.Size([1000])
CPU times: user 3.47 s, sys: 2.48 s, total: 5.96 s
Wall time: 3.49 s


In [14]:
%%time 

graph_list = [
    NucleiData(X, C_x, A, E) for (X, C_x, A, E) in 
    zip(X.unbind(), C_x.unbind(), A.unbind(), E.unbind())
]



graph_list = [
    clear_iso_nodes(graph)
    for graph in graph_list
]

explainee.eval()
test = [
    print(graph)
    for graph in graph_list
]

from torch_geometric.loader import DataLoader

graph_loader = DataLoader(graph_list, batch_size=100, shuffle=False)

NucleiData(x=[50, 11], edge_index=[2, 1276], edge_attr=[1276, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1218], edge_attr=[1218, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1295], edge_attr=[1295, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1183], edge_attr=[1183, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1225], edge_attr=[1225, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1225], edge_attr=[1225, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1243], edge_attr=[1243, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1239], edge_attr=[1239, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1245], edge_attr=[1245, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1263], edge_attr=[1263, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1230], edge_attr=[1230, 3], cell_type=[50])
NucleiData(x=[50, 11], edge_index=[2, 1250], edge_attr=[1250, 3], cell_type=[50])
NucleiData(x=[50

In [76]:
model = load_model(path = "../data/trained/epoch_263.pt",
    device=device)

In [58]:
model.eval()
criterion = nn.BCELoss()
for _, batch in enumerate(graph_loader):
    batch.to(torch.device(0))
    model.eval()
    explainee_pred = torch.softmax(model(batch), dim=0).cpu()
    #target_batched = torch.tensor([0.0, 1.0]).repeat(100)
    # print(criterion(explainee_pred, batch.y.squeeze()))
    print(explainee_pred.split(1, dim=0))


(tensor([0.9830], grad_fn=<SplitBackward0>), tensor([0.0170], grad_fn=<SplitBackward0>))
(tensor([0.9900], grad_fn=<SplitBackward0>), tensor([0.0100], grad_fn=<SplitBackward0>))
(tensor([0.9817], grad_fn=<SplitBackward0>), tensor([0.0183], grad_fn=<SplitBackward0>))
(tensor([0.9844], grad_fn=<SplitBackward0>), tensor([0.0156], grad_fn=<SplitBackward0>))
(tensor([0.9854], grad_fn=<SplitBackward0>), tensor([0.0146], grad_fn=<SplitBackward0>))
(tensor([0.9901], grad_fn=<SplitBackward0>), tensor([0.0099], grad_fn=<SplitBackward0>))
(tensor([0.9810], grad_fn=<SplitBackward0>), tensor([0.0190], grad_fn=<SplitBackward0>))
(tensor([0.9880], grad_fn=<SplitBackward0>), tensor([0.0120], grad_fn=<SplitBackward0>))
(tensor([0.9868], grad_fn=<SplitBackward0>), tensor([0.0132], grad_fn=<SplitBackward0>))
(tensor([0.9864], grad_fn=<SplitBackward0>), tensor([0.0136], grad_fn=<SplitBackward0>))


In [29]:
%%time 

import pygmtools as pygm
pygm.set_backend('pytorch')

batched_A = torch.randn((2, 10, 10))
sparse_A = pygm.utils.dense_to_sparse(batched_A)[0]
print(sparse_A[1].T)


tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
         2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4,
         4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7,
         7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9,
         9, 9, 9, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3,
         4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7,
         8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1,
         2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5,
         6, 7, 8, 9]])
CPU times: user 2.43 ms, sys: 353 µs, total: 2.78 ms
Wall time: 2.08 ms
